<a href="https://colab.research.google.com/github/Mahendra2409/PyBlender/blob/main/Colab_Script/gcs_to_drive_transfer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 GCS → Google Drive Transfer

Transfer all rendered images from `gs://pyblender-render-farm/RenderImages/` to `MyDrive/PyBlender/Compare/`

**Auth Strategy:**
- **GCS**: Service account key (`pyblender-e37593034bc1.json`)
- **Drive**: Colab's native Google Drive mount (your Drive Gmail account)

**Drive folder structure:**
```
PyBlender/Compare/
├── boy_01_PC_v2/
│   ├── viridis_colormap/
│   │   ├── boy01.png
│   │   ├── boy01_noisy.png
│   │   └── ...
│   ├── inferno_colormap/
│   └── ... (168 colormap folders)
├── boy_02_pc_v2/
└── ...
```

## Cell 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Cell 2 — Install Dependencies

In [2]:
!pip install -q google-cloud-storage

## Cell 3 — Load GCS Service Account Key

Load the service account JSON key from Colab **Secrets**.
Ensure you have a secret named `GCS_SERVICE_ACCOUNT_KEY` with the content of `pyblender.json`.

In [3]:
from google.colab import userdata
import os

GCS_KEY_PATH = '/tmp/pyblender.json'

if os.path.exists(GCS_KEY_PATH):
    print(f'✓ Key already exists at {GCS_KEY_PATH}')
else:
    try:
        key_content = userdata.get('GCS_SERVICE_ACCOUNT_KEY')
        with open(GCS_KEY_PATH, 'w') as f:
            f.write(key_content)
        print(f'✓ Saved secret to {GCS_KEY_PATH}')
    except userdata.SecretNotFoundError:
        print("✗ Secret 'GCS_SERVICE_ACCOUNT_KEY' not found!")
        print("Please add it to the 'Secrets' tab (key icon) on the left sidebar.")

✓ Saved secret to /tmp/pyblender.json


## Cell 4 — Transfer ALL Files from GCS → Drive (FAST)

**Two-phase approach for maximum speed:**
1. **Phase 1**: 32 concurrent threads download from GCS to Colab local SSD
2. **Phase 2**: Batch copy from local SSD to Drive mount

- **Resume-safe**: Skips files that already exist on Drive
- **Set `FORCE_OVERWRITE = True`** to re-download everything

In [4]:
import os
import time
import shutil
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from google.cloud import storage
from collections import defaultdict

# ─── Configuration ─────────────────────────────────────────
GCS_KEY_PATH = '/tmp/pyblender-e37593034bc1.json'
BUCKET_NAME = 'pyblender-render-farm'
GCS_BASE_PATH = 'RenderImages'
DRIVE_BASE_PATH = '/content/drive/MyDrive/PyBlender/Compare'
LOCAL_TEMP_DIR = '/content/temp_gcs_download'
FORCE_OVERWRITE = False
MAX_WORKERS = 32
# ───────────────────────────────────────────────────────────

def sizeof_fmt(num_bytes):
    for unit in ['B', 'KB', 'MB', 'GB']:
        if abs(num_bytes) < 1024.0:
            return f'{num_bytes:.1f} {unit}'
        num_bytes /= 1024.0
    return f'{num_bytes:.1f} TB'


class ProgressTracker:
    def __init__(self, total):
        self.total = total
        self.transferred = 0
        self.skipped = 0
        self.failed = 0
        self.bytes_transferred = 0
        self.lock = threading.Lock()
        self.failed_files = []
        self.start_time = time.time()

    def record_transfer(self, size):
        with self.lock:
            self.transferred += 1
            self.bytes_transferred += size
            self._maybe_print()

    def record_skip(self):
        with self.lock:
            self.skipped += 1
            self._maybe_print()

    def record_fail(self, path, error):
        with self.lock:
            self.failed += 1
            self.failed_files.append((path, error))
            print(f'   ✗ FAILED: {path} — {error}')

    def _maybe_print(self):
        done = self.transferred + self.skipped + self.failed
        if done % 50 == 0 or done == self.total:
            elapsed = time.time() - self.start_time
            rate = self.transferred / elapsed if elapsed > 0 else 0
            remaining = self.total - done
            eta = remaining / rate if rate > 0 else 0
            print(
                f'   [{done}/{self.total}] '
                f'✓ {self.transferred} downloaded, ⏭ {self.skipped} skipped, ✗ {self.failed} failed '
                f'| {sizeof_fmt(self.bytes_transferred)} | {rate:.1f} files/s | ETA: {eta:.0f}s'
            )


def download_blob(blob, local_path, drive_path, tracker):
    relative_path = blob.name[len(GCS_BASE_PATH) + 1:]
    if not FORCE_OVERWRITE and os.path.exists(drive_path):
        tracker.record_skip()
        return
    try:
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        blob.download_to_filename(local_path)
        tracker.record_transfer(blob.size or 0)
    except Exception as e:
        tracker.record_fail(relative_path, str(e))


def transfer_gcs_to_drive():
    print('🔗 Connecting to GCS...')
    client = storage.Client.from_service_account_json(GCS_KEY_PATH)
    bucket = client.bucket(BUCKET_NAME)

    try:
        next(bucket.list_blobs(max_results=1, prefix=GCS_BASE_PATH + '/'))
        print(f'✓ Connected to bucket: {BUCKET_NAME}')
    except StopIteration:
        print('⚠ Bucket is empty or prefix has no files')
        return
    except Exception as e:
        print(f'✗ Failed to access bucket: {e}')
        return

    print(f'\n📋 Listing all files under gs://{BUCKET_NAME}/{GCS_BASE_PATH}/...')
    all_blobs = []
    total_size = 0
    for blob in bucket.list_blobs(prefix=GCS_BASE_PATH + '/'):
        if blob.name.endswith('/'):
            continue
        all_blobs.append(blob)
        total_size += blob.size or 0

    print(f'   Found {len(all_blobs)} files ({sizeof_fmt(total_size)})')
    if not all_blobs:
        print('Nothing to transfer!')
        return

    pc_types = defaultdict(lambda: defaultdict(int))
    for blob in all_blobs:
        parts = blob.name[len(GCS_BASE_PATH) + 1:].split('/')
        if len(parts) >= 2:
            pc_types[parts[0]][parts[1]] += 1
    print(f'\n📂 Point Cloud Types found:')
    for pc_type, colormaps in sorted(pc_types.items()):
        total_files = sum(colormaps.values())
        print(f'   ├── {pc_type}: {len(colormaps)} colormaps, {total_files} files')

    if os.path.exists(LOCAL_TEMP_DIR):
        shutil.rmtree(LOCAL_TEMP_DIR)
    os.makedirs(LOCAL_TEMP_DIR, exist_ok=True)
    os.makedirs(DRIVE_BASE_PATH, exist_ok=True)

    # ── PHASE 1: Concurrent download GCS → Local SSD ─────
    print(f'\n🚀 PHASE 1: Downloading from GCS → local SSD ({MAX_WORKERS} threads)...\n')
    tracker = ProgressTracker(len(all_blobs))

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = []
        for blob in all_blobs:
            relative_path = blob.name[len(GCS_BASE_PATH) + 1:]
            local_path = os.path.join(LOCAL_TEMP_DIR, relative_path)
            drive_path = os.path.join(DRIVE_BASE_PATH, relative_path)
            futures.append(
                executor.submit(download_blob, blob, local_path, drive_path, tracker)
            )
        for future in as_completed(futures):
            try:
                future.result()
            except Exception as e:
                print(f'   ✗ Unexpected thread error: {e}')

    phase1_elapsed = time.time() - tracker.start_time
    print(f'\n   Phase 1 done: {tracker.transferred} files in {phase1_elapsed:.1f}s')

    if tracker.transferred == 0:
        print('   Nothing new to copy to Drive.')
        shutil.rmtree(LOCAL_TEMP_DIR, ignore_errors=True)
        return tracker.transferred, tracker.skipped, tracker.failed

    # ── PHASE 2: Batch copy Local SSD → Drive ────────────
    print(f'\n📦 PHASE 2: Copying {tracker.transferred} files to Google Drive...')
    phase2_start = time.time()
    copied = 0
    copy_failed = 0
    for root, dirs, files_list in os.walk(LOCAL_TEMP_DIR):
        for f in files_list:
            src = os.path.join(root, f)
            relative = os.path.relpath(src, LOCAL_TEMP_DIR)
            dst = os.path.join(DRIVE_BASE_PATH, relative)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            try:
                shutil.copy2(src, dst)
                copied += 1
                if copied % 100 == 0:
                    print(f'   Copied {copied}/{tracker.transferred} files to Drive...')
            except Exception as e:
                copy_failed += 1
                print(f'   ✗ Copy failed: {relative} — {e}')

    phase2_elapsed = time.time() - phase2_start
    print(f'   Phase 2 done: {copied} files copied in {phase2_elapsed:.1f}s')

    shutil.rmtree(LOCAL_TEMP_DIR, ignore_errors=True)

    total_elapsed = time.time() - tracker.start_time
    sep = '=' * 60
    print(f'\n{sep}')
    print('✅ TRANSFER COMPLETE')
    print(sep)
    print(f'   Downloaded  : {tracker.transferred} files ({sizeof_fmt(tracker.bytes_transferred)})')
    print(f'   Copied      : {copied} files to Drive')
    print(f'   Skipped     : {tracker.skipped} files (already on Drive)')
    print(f'   Failed      : {tracker.failed} download + {copy_failed} copy failures')
    print(f'   Phase 1     : {phase1_elapsed:.1f}s (GCS → local, {MAX_WORKERS} threads)')
    print(f'   Phase 2     : {phase2_elapsed:.1f}s (local → Drive)')
    print(f'   Total time  : {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)')
    if total_elapsed > 0 and tracker.transferred > 0:
        print(f'   Avg speed   : {tracker.transferred/total_elapsed:.1f} files/s')
    print(f'   Drive path  : {DRIVE_BASE_PATH}')
    print(sep)

    if tracker.failed_files:
        print('\n⚠ Failed downloads:')
        for path, error in tracker.failed_files:
            print(f'   • {path}: {error}')

    return tracker.transferred, tracker.skipped, tracker.failed


transfer_gcs_to_drive()

🔗 Connecting to GCS...
⚠ Bucket is empty or prefix has no files


## Cell 5 — Verify Transfer

Check what ended up on Drive

In [5]:
import os
from collections import defaultdict

DRIVE_BASE_PATH = '/content/drive/MyDrive/PyBlender/Compare'

print('🔍 Verifying Drive contents...\n')

stats = defaultdict(lambda: defaultdict(int))
total_files = 0
total_size = 0

for root, dirs, files_list in os.walk(DRIVE_BASE_PATH):
    for f in files_list:
        fpath = os.path.join(root, f)
        rel = os.path.relpath(fpath, DRIVE_BASE_PATH)
        parts = rel.split(os.sep)
        if len(parts) >= 2:
            stats[parts[0]][parts[1]] += 1
        total_files += 1
        total_size += os.path.getsize(fpath)

print(f'📂 {DRIVE_BASE_PATH}')
print(f'   Total: {total_files} files ({total_size / (1024*1024):.1f} MB)\n')

for pc_type in sorted(stats):
    colormaps = stats[pc_type]
    files_count = sum(colormaps.values())
    print(f'   📁 {pc_type}/')
    print(f'      {len(colormaps)} colormap folders, {files_count} files')

print(f'\n✅ Verification complete!')

🔍 Verifying Drive contents...

📂 /content/drive/MyDrive/PyBlender/Compare
   Total: 545 files (736.6 MB)

   📁 boy_01_PC_v2/
      50 colormap folders, 545 files

✅ Verification complete!


## Cell 5.1 — Visualization Configuration

Edit these settings before running the visualization cell below.

| Setting | Description |
|---------|-------------|
| `DRIVE_BASE_PATH` | Root Drive path containing `RenderImages/` |
| `PC_TYPE` | Point cloud dataset subfolder name |
| `COLORMAPS` | List of colormaps to visualize (empty = auto-detect all) |
| `FORCE_OVERWRITE` | Re-generate even if output already exists |
| `SAVE_WORKERS` | Parallel threads for saving PNGs to Drive |
| `OUTPUT_DPI` | Output image DPI (300 = publication quality) |

In [ ]:
# @title Cell 5.1 — Visualization Configuration

# ═══════════════════════════════════════════════════════════
# VISUALIZATION CONFIGURATION
# ═══════════════════════════════════════════════════════════

VIZ_CONFIG = {
    # --- Paths ---
    "DRIVE_BASE_PATH": "/content/drive/MyDrive/PyBlender_Render_Farm",
    "PC_TYPE": "boy_01_PC_v2",

    # --- Colormaps (empty list = auto-detect from Drive) ---
    "COLORMAPS": [],

    # --- Output Controls ---
    "FORCE_OVERWRITE": False,   # Set True to re-generate existing comparisons
    "OUTPUT_DPI": 300,          # 300 = publication quality

    # --- Performance ---
    "SAVE_WORKERS": 4,          # Parallel threads for saving to Drive
}

# ═══════════════════════════════════════════════════════════
# LABEL MAPPING (filename pattern → display label + order)
# ═══════════════════════════════════════════════════════════

LABEL_RULES = [
    (lambda n: n.startswith("boy01_noisy"), "Noisy", 0),
    (lambda n: "bilateral" in n,           "BF",    1),
    (lambda n: "wlop" in n,                "WLOP",  2),
    (lambda n: "ad_" in n,                 "AD",    3),
    (lambda n: "dmr" in n,                 "DMR",   4),
    (lambda n: "score" in n,               "Score", 5),
    (lambda n: "iterpfn" in n,             "IterPFN",      6),
    (lambda n: "straightpcf" in n,         "StraightPCF",  7),
    (lambda n: "delnoise" in n,            "De(l)Noise",   8),
    (lambda n: "ours" in n,                "Ours",  9),
    (lambda n: n.startswith("boy01."),      "GT",   10),
]

print("✅ Visualization configuration loaded.")
print(f"   PC_TYPE       : {VIZ_CONFIG['PC_TYPE']}")
print(f"   COLORMAPS     : {'auto-detect' if not VIZ_CONFIG['COLORMAPS'] else str(len(VIZ_CONFIG['COLORMAPS'])) + ' specified'}")
print(f"   FORCE_OVERWRITE: {VIZ_CONFIG['FORCE_OVERWRITE']}")
print(f"   OUTPUT_DPI    : {VIZ_CONFIG['OUTPUT_DPI']}")
print(f"   SAVE_WORKERS  : {VIZ_CONFIG['SAVE_WORKERS']}")

## Cell 6 — Generate Comparison Visualizations (Optimized)

**Speed optimizations over original:**
- Pre-loads & caches all images as numpy arrays (eliminates repeated PIL decode)
- Pre-computes colorbar gradient once (reused across all colormaps)
- Parallel saves to Drive via `ThreadPoolExecutor`
- Skips already-generated comparisons (unless `FORCE_OVERWRITE = True`)

**Visualization quality is identical** (same figsize, dpi, fonts, layout, cropping).

In [ ]:
# @title Cell 6 — Generate Comparison Visualizations (Optimized)
import os
import time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Pull config from the configuration cell ──
DRIVE_BASE_PATH = VIZ_CONFIG["DRIVE_BASE_PATH"]
PC_TYPE         = VIZ_CONFIG["PC_TYPE"]
COLORMAPS       = list(VIZ_CONFIG["COLORMAPS"])
FORCE_OVERWRITE = VIZ_CONFIG["FORCE_OVERWRITE"]
OUTPUT_DPI      = VIZ_CONFIG["OUTPUT_DPI"]
SAVE_WORKERS    = VIZ_CONFIG["SAVE_WORKERS"]

# ── Label helper using LABEL_RULES from config ──
def get_label_and_sort_key(filename):
    name = filename.lower()
    for rule_fn, label, order in LABEL_RULES:
        if rule_fn(name):
            return label, order
    return filename.split('.')[0].title(), 99

# ── Auto-detect colormaps if list is empty ──
if not COLORMAPS:
    print("COLORMAPS list is empty. Auto-detecting from Drive...")
    base_render_dir = os.path.join(DRIVE_BASE_PATH, "RenderImages", PC_TYPE)
    if os.path.exists(base_render_dir):
        COLORMAPS = [
            folder.replace("_colormap", "")
            for folder in os.listdir(base_render_dir)
            if os.path.isdir(os.path.join(base_render_dir, folder)) and folder.endswith("_colormap")
        ]
        if COLORMAPS:
            print(f"✅ Detected {len(COLORMAPS)} colormaps\n")
        else:
            print("❌ No colormap directories found. Exiting.")
    else:
        print(f"❌ Base render directory not found: {base_render_dir}")

if not COLORMAPS:
    raise SystemExit("No colormaps to process.")

# ── Create output directory ──
compare_dir = os.path.join(DRIVE_BASE_PATH, "Compare", PC_TYPE)
os.makedirs(compare_dir, exist_ok=True)

# ── Pre-compute the colorbar gradient (reused for every colormap) ──
GRADIENT = np.linspace(1, 0, 256).reshape(256, 1)

# ── Helper: load + crop + convert to numpy array ──
def load_and_crop(img_path):
    img = Image.open(img_path)
    bbox = img.getbbox()
    if bbox:
        img = img.crop(bbox)
    return np.asarray(img)

# ── Helper: save figure to disk (runs in thread pool) ──
def save_figure(fig, out_path, dpi):
    fig.savefig(out_path, bbox_inches='tight', pad_inches=0.2, dpi=dpi, facecolor='white')
    plt.close(fig)
    return out_path

# ── Main generation loop ──
t_start = time.time()
generated = 0
skipped = 0
save_futures = []

executor = ThreadPoolExecutor(max_workers=SAVE_WORKERS)

try:
    for idx, colormap in enumerate(COLORMAPS):
        render_dir = os.path.join(DRIVE_BASE_PATH, "RenderImages", PC_TYPE, f"{colormap}_colormap")
        if not os.path.exists(render_dir):
            continue

        out_path = os.path.join(compare_dir, f"{colormap}_comparison.png")

        # Skip if already exists and not forcing overwrite
        if not FORCE_OVERWRITE and os.path.exists(out_path):
            skipped += 1
            continue

        image_files = [f for f in os.listdir(render_dir) if f.endswith('.png') and 'colormap' not in f]
        if not image_files:
            continue

        image_files.sort(key=lambda x: get_label_and_sort_key(x)[1])
        n_images = len(image_files)

        # Pre-load all images as numpy arrays (batch I/O before plotting)
        loaded_images = []
        labels = []
        for img_name in image_files:
            img_path = os.path.join(render_dir, img_name)
            loaded_images.append(load_and_crop(img_path))
            label, _ = get_label_and_sort_key(img_name)
            labels.append(label)

        # Create figure (identical layout to original)
        fig, axes = plt.subplots(1, n_images + 1, figsize=(n_images * 3.5, 6), gridspec_kw={'width_ratios': [1]*n_images + [0.15]})
        fig.subplots_adjust(wspace=0.02)
        fig.suptitle(f"Colormap: {colormap.upper()}", fontsize=28, fontweight='bold', fontfamily='serif', y=1.05)

        # Plot pre-loaded images
        for i in range(n_images):
            axes[i].imshow(loaded_images[i])
            axes[i].axis('off')
            axes[i].text(0.5, -0.05, labels[i], size=18, ha="center", va="top", transform=axes[i].transAxes, fontfamily='serif')

        # Draw the Vertical Colorbar (reusing pre-computed gradient)
        axes[-1].imshow(GRADIENT, aspect='auto', cmap=colormap)
        axes[-1].axis('off')

        # Submit save to thread pool (Drive I/O happens in background)
        save_futures.append(executor.submit(save_figure, fig, out_path, OUTPUT_DPI))
        generated += 1

        # Free image arrays from memory
        del loaded_images, labels

        # Progress update every 25 colormaps
        if generated % 25 == 0:
            elapsed = time.time() - t_start
            rate = generated / elapsed if elapsed > 0 else 0
            print(f"   [{generated} generated, {skipped} skipped] {rate:.1f} comparisons/s")

    # Wait for all background saves to complete
    for future in as_completed(save_futures):
        try:
            path = future.result()
        except Exception as e:
            print(f"   ✗ Save failed: {e}")

finally:
    executor.shutdown(wait=True)
    plt.close('all')

# ── Summary ──
total_time = time.time() - t_start
sep = '=' * 55
print(f"\n{sep}")
print("✅ VISUALIZATION COMPLETE")
print(sep)
print(f"   Generated  : {generated} comparison images")
print(f"   Skipped    : {skipped} (already exist)")
print(f"   Total time : {total_time:.1f}s ({total_time/60:.1f} min)")
if generated > 0:
    print(f"   Avg speed  : {generated/total_time:.1f} comparisons/s")
print(f"   Output dir : {compare_dir}")
print(sep)

In [ ]:
# @title Cell 7-Generate PDF Catalog (Montserrat Headings)
import os
import urllib.request
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image, ImageFile

# Tell Pillow to ignore slightly truncated files caused by Drive syncing
ImageFile.LOAD_TRUNCATED_IMAGES = True

# --- NEW: Fetch and Load Montserrat Font ---
font_url = "https://github.com/JulietaUla/Montserrat/raw/master/fonts/ttf/Montserrat-Bold.ttf"
font_path = "/content/Montserrat-Bold.ttf"

if not os.path.exists(font_path):
    print("Downloading Montserrat font...")
    urllib.request.urlretrieve(font_url, font_path)

# Create a specific FontProperties object to pass to our titles
heading_font = fm.FontProperties(fname=font_path, size=30)

# 1. Pull settings from config
try:
    from config import CONFIG
    DRIVE_BASE_PATH = CONFIG["DRIVE_BASE_PATH"]
    PC_TYPE = CONFIG["PC_TYPE"]
except ImportError:
    # Fallback
    DRIVE_BASE_PATH = "/content/drive/MyDrive/PyBlender_Render_Farm"
    PC_TYPE = "boy_01_PC_v2"

compare_dir = os.path.join(DRIVE_BASE_PATH, "Compare", PC_TYPE)
pdf_path = os.path.join(DRIVE_BASE_PATH, "Compare", f"{PC_TYPE}_Colormap_Catalog.pdf")

# 2. Gather Comparison Images
image_files = [f for f in os.listdir(compare_dir) if f.endswith('_comparison.png')]
image_files.sort()

if not image_files:
    print(f"❌ No comparison images found in {compare_dir}.")
else:
    print(f"Found {len(image_files)} comparison images. Building PDF catalog...")

    # 3. Layout Settings
    IMAGES_PER_PAGE = 6

    # 4. Create multi-page PDF
    with PdfPages(pdf_path) as pdf:
        num_pages = (len(image_files) + IMAGES_PER_PAGE - 1) // IMAGES_PER_PAGE

        for page_idx in range(num_pages):
            fig, axes = plt.subplots(IMAGES_PER_PAGE, 1, figsize=(16, 24))

            if IMAGES_PER_PAGE == 1:
                axes = [axes]

            for i in range(IMAGES_PER_PAGE):
                img_idx = page_idx * IMAGES_PER_PAGE + i
                ax = axes[i]

                if img_idx < len(image_files):
                    img_path = os.path.join(compare_dir, image_files[img_idx])
                    img = Image.open(img_path)
                    ax.imshow(img)

                ax.axis('off')

            # --- APPLIED FONT: Using the 'fontproperties' argument ---
            # title_text = f"{PC_TYPE.replace('_', ' ').title()} - Colormap Catalog (Page {page_idx + 1})"
            # fig.suptitle(title_text, fontproperties=heading_font, color='#222222')

            # plt.tight_layout()
            # plt.subplots_adjust(top=0.92, hspace=0.1)

            pdf.savefig(fig, bbox_inches='tight', facecolor='white')
            plt.close(fig)

    print(f"✅ PDF Catalog successfully saved to: {pdf_path}")

## Cell 6 (Optional) — Cross-Check: Find Missing Files

Compares bucket contents against Drive to ensure **nothing was left behind**

In [ ]:
import os
from google.cloud import storage

GCS_KEY_PATH = '/tmp/pyblender-e37593034bc1.json'
BUCKET_NAME = 'pyblender-render-farm'
GCS_BASE_PATH = 'RenderImages'
DRIVE_BASE_PATH = '/content/drive/MyDrive/PyBlender/Compare'

client = storage.Client.from_service_account_json(GCS_KEY_PATH)
bucket = client.bucket(BUCKET_NAME)

missing = []
matched = 0

for blob in bucket.list_blobs(prefix=GCS_BASE_PATH + '/'):
    if blob.name.endswith('/'):
        continue
    relative_path = blob.name[len(GCS_BASE_PATH) + 1:]
    drive_path = os.path.join(DRIVE_BASE_PATH, relative_path)
    if os.path.exists(drive_path):
        matched += 1
    else:
        missing.append(relative_path)

print(f'✓ Matched on Drive: {matched}')
print(f'✗ Missing from Drive: {len(missing)}')

if missing:
    print(f'\nMissing files:')
    for m in missing[:50]:
        print(f'   • {m}')
    if len(missing) > 50:
        print(f'   ... and {len(missing) - 50} more')
else:
    print(f'\n✅ ALL bucket files are present on Drive! Nothing left behind.')